# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Hyperparameters


In [ ]:
BATCH_SIZE = 4

# Wether to use the COCO pretrained Faster R-CNN
PRETRAINED = True

# Wether to use the AGAR dataset for pretraining
AGAR_TRAINING = False
EPOCHS_AGAR = 20

EPOCHS_OWN = 200
KFOLDS_OWN = 5

# Max norm for gradient clipping
MAX_NORM = 1

## Logging code


In [ ]:
import csv
from datetime import datetime
from pathlib import Path

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()


start_time_str = datetime.now().strftime("%Y%m%d_%H%M")
options_str = f"{"COCO" if PRETRAINED else "NOCOCO"}_{"AGAR" if AGAR_TRAINING else "NOAGAR"}"

artifact_path = Path("artifacts") / f"{start_time_str}_{options_str}"
artifact_path.mkdir(exist_ok=True, parents=True)

def setup_csv(file_name: str):
    """Set up the CSV file with the appropriate headers."""
    with open(artifact_path / file_name, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "stage",
                "fold",
                "epoch",
                "train_loss",
                "train_loss_classifier",
                "train_loss_box_reg",
                "train_loss_objectness",
                "train_loss_rpn_box_reg",
                "val_loss",
                "val_loss_classifier",
                "val_loss_box_reg",
                "val_loss_objectness",
                "val_loss_rpn_box_reg",
                "val_avg_precision_iou_50",
                "val_avg_recall_iou_50",
                "val_mae",
                "val_smape",
            ]
        )

    def append_to_csv(row):
        """Append a row of data to the CSV file."""
        with open(artifact_path / file_name, "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(row)

    return append_to_csv

## Training code


In [ ]:
from engine import evaluate, train_one_epoch
from torch.optim import AdamW
import torch
import utils
from dataset_agar import AgarBBDataset
from model import get_model_instance_segmentation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")


def train_model(
    stage: str,
    train_set: torch.utils.data.Dataset,
    val_set: torch.utils.data.Dataset = None,
    num_epochs=20,
    model=None,
    save_scores: callable = None,
    fold: int = None,
):
    should_val = val_set is not None

    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        num_workers=4,
        pin_memory=True,
        shuffle=True,
        collate_fn=utils.collate_fn,
    )

    if model is None:
        model = get_model_instance_segmentation(
            AgarBBDataset.num_classes,
            pretrained=PRETRAINED,
        ).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            scaler=scaler,
            print_freq=10,
            max_norm=MAX_NORM,
        )

        avg_train_loss_classifier = sum(
            [loss["loss_classifier"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_box_reg = sum(
            [loss["loss_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_objectness = sum(
            [loss["loss_objectness"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_rpn_box_reg = sum(
            [loss["loss_rpn_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss = sum(
            [
                (
                    loss["loss_classifier"]
                    + loss["loss_box_reg"]
                    + loss["loss_objectness"]
                    + loss["loss_rpn_box_reg"]
                )
                for loss in train_losses
            ]
        ) / len(train_losses)

        # check loss on validation set
        if should_val:
            val_loader = torch.utils.data.DataLoader(
                val_set,
                batch_size=BATCH_SIZE,
                num_workers=4,
                pin_memory=True,
                shuffle=False,
                collate_fn=utils.collate_fn,
            )

            with torch.no_grad():
                val_losses = train_one_epoch(
                    model,
                    optimizer,
                    val_loader,
                    device,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_classifier = sum(
                    [loss["loss_classifier"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_box_reg = sum(
                    [loss["loss_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_objectness = sum(
                    [loss["loss_objectness"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_rpn_box_reg = sum(
                    [loss["loss_rpn_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss = sum(
                    [
                        (
                            loss["loss_classifier"]
                            + loss["loss_box_reg"]
                            + loss["loss_objectness"]
                            + loss["loss_rpn_box_reg"]
                        )
                        for loss in val_losses
                    ]
                ) / len(val_losses)


                # evaluate on the validation dataset
                coco_evaluator, mae, smape = evaluate(model, val_loader, device=device)

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

        if save_scores is not None:
            save_scores(
                [
                    stage,
                    float(fold) if fold is not None else "N/A",
                    float(epoch),
                    float(avg_train_loss),
                    float(avg_train_loss_classifier),
                    float(avg_train_loss_box_reg),
                    float(avg_train_loss_objectness),
                    float(avg_train_loss_rpn_box_reg),
                    float(avg_val_loss) if should_val else "N/A",
                    float(avg_val_loss_classifier) if should_val else "N/A",
                    float(avg_val_loss_box_reg) if should_val else "N/A",
                    float(avg_val_loss_objectness) if should_val else "N/A",
                    float(avg_val_loss_rpn_box_reg) if should_val else "N/A",
                    float(avg_precision_iou_50) if should_val else "N/A",
                    float(avg_recall_iou_50) if should_val else "N/A",
                    float(mae) if should_val else "N/A",
                    float(smape) if should_val else "N/A",
                ],
            )
            
            prefix = f"{stage}f{f" {fold}" if fold is not None else ""}"

            writer.add_scalar(f"{prefix} avg_train_loss_classifier", avg_train_loss_classifier, epoch)
            writer.add_scalar(f"{prefix} avg_train_loss_box_reg", avg_train_loss_box_reg, epoch)
            writer.add_scalar(f"{prefix} avg_train_loss_objectness", avg_train_loss_objectness, epoch)
            writer.add_scalar(f"{prefix} avg_train_loss_rpn_box_reg", avg_train_loss_rpn_box_reg, epoch)
            writer.add_scalar(f"{prefix} avg_train_loss", avg_train_loss, epoch)

            writer.add_scalar(f"{prefix} avg_val_loss_classifier", avg_val_loss_classifier, epoch)
            writer.add_scalar(f"{prefix} avg_val_loss_box_reg", avg_val_loss_box_reg, epoch)
            writer.add_scalar(f"{prefix} avg_val_loss_objectness", avg_val_loss_objectness, epoch)
            writer.add_scalar(f"{prefix} avg_val_loss_rpn_box_reg", avg_val_loss_rpn_box_reg, epoch)
            writer.add_scalar(f"{prefix} avg_val_loss", avg_val_loss, epoch)

            writer.add_scalar(f"{prefix} avg_precision_iou_50", avg_precision_iou_50, epoch)
            writer.add_scalar(f"{prefix} avg_recall_iou_50", avg_recall_iou_50, epoch)
            writer.add_scalar(f"{prefix} mae", mae, epoch)
            writer.add_scalar(f"{prefix} smape", smape, epoch)
            
            writer.flush()

    return model

## Training on the AGAR dataset

Simple training and validation run on the high-resolution images of the AGAR
dataset.


In [ ]:
import json
from dataset_transforms import get_transforms

write_row = setup_csv(f"metrics.csv")

if AGAR_TRAINING:
    print("Training on AGAR dataset\n" + "-" * 10 + "\n")

    # The box data for these IDs of the AGAR dataset have negative x values for the bounding boxes.
    faulty_ids = ["5794", "11988"]

    with open("../AGAR_dataset/training_lists/higher_resolution_train.txt") as f:
        train_list = json.load(f)
        train_list = [
            int(id) for id in train_list if id.isdigit() and id not in faulty_ids
        ]

    with open("../AGAR_dataset/training_lists/higher_resolution_val.txt") as f:
        val_list = json.load(f)
        val_list = [int(id) for id in val_list if id.isdigit() and id not in faulty_ids]

    print(f"Training on {len(train_list)} images.")
    print(f"Validating on {len(val_list)} images.")

    train_dataset = AgarBBDataset("../AGAR_dataset/dataset", train_list, is_train=True)
    val_dataset = AgarBBDataset("../AGAR_dataset/dataset", val_list, is_train=False)

    agar_trained_model = train_model(
        "agar_pretraining",
        train_set=train_dataset,
        val_set=val_dataset,
        num_epochs=EPOCHS_AGAR,
        save_scores=write_row,
    )

    agar_model_path = str(artifact_path / "agar_trained.pth")

    torch.save(agar_trained_model.state_dict(), agar_model_path)

## K-fold cross validation on own dataset

K different models are trained and validated here


In [ ]:
from dataset_photobox import PhotoboxBBDataset


def get_base_model():
    if AGAR_TRAINING:
        model = get_model_instance_segmentation(
            PhotoboxBBDataset.num_classes, pretrained=False
        ).to(device)
        model.load_state_dict(torch.load(agar_model_path))
        return model
    else:
        return get_model_instance_segmentation(
            PhotoboxBBDataset.num_classes, pretrained=PRETRAINED
        ).to(device)

In [ ]:
from sklearn.model_selection import KFold


kfold = KFold(n_splits=KFOLDS_OWN, shuffle=True)

full_train_dataset = PhotoboxBBDataset(
    "../training_data", is_train=True, patch_augmentation=True
)

full_val_dataset = PhotoboxBBDataset("../training_data", is_train=False)

for fold, (train_ids, val_ids) in enumerate(kfold.split(full_train_dataset)):
    print(f"Training fold {fold}\n" + "-" * 10 + "\n")

    train_dataset = torch.utils.data.Subset(full_train_dataset, train_ids)
    val_dataset = torch.utils.data.Subset(full_val_dataset, val_ids)

    base_model = get_base_model()

    fold_model = train_model(
        "kfold",
        train_set=train_dataset,
        val_set=val_dataset,
        num_epochs=EPOCHS_OWN,
        model=base_model,
        save_scores=write_row,
        fold=fold,
    )

    torch.save(
        fold_model.state_dict(),
        str(artifact_path / f"model_fold_{fold}.pth"),
    )

## Final training run

The final training run on all of the available training data.


In [ ]:
print(f"Final training run\n" + "-" * 10 + "\n")

base_model = get_base_model()

final_model = train_model(
    "final_training",
    train_set=full_train_dataset,
    model=base_model,
    num_epochs=EPOCHS_OWN,
    save_scores=write_row,
)

# save the model
torch.save(
    final_model.state_dict(),
    str(artifact_path / f"final_model.pth"),
)

## Evaluation

Evaluate the final model on key metrics.


In [ ]:
print(f"Evaluating on test set\n" + "-" * 10 + "\n")

test_dataset = PhotoboxBBDataset("../test_data", is_train=False)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

coco_evaluator, mae, smape = evaluate(final_model, test_loader, device=device)

stats = coco_evaluator.coco_eval["bbox"].stats.tolist()

with open(artifact_path / f"test_results.txt", "w") as f:
    f.write(f"Average absolute error: {mae}\n")
    f.write(f"Average relative error: {smape}\n")

    f.write(json.dumps(stats, indent=4))

writer.close()